In [2]:
# filter dataset
!python MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224753
--- HTML Removed			--> Rows: 224753
--- Rows will remain true-cased		--> Rows: 224753
--- Rows with Empty Cells Deleted	--> Rows: 224753
--- Source Saved: ./en-zh.zh-filtered-wsd.zh
--- Target Saved: ./en-zh.en-filtered-wsd.en


In [3]:
def create_mapping(unshuffled_file, shuffled_file):
    with open(unshuffled_file, 'r', encoding='utf-8') as file:
        unshuffled_lines = [line.strip() for line in file.readlines()]

    with open(shuffled_file, 'r', encoding='utf-8') as file:
        shuffled_lines = [line.strip() for line in file.readlines()]

    line_to_index_shuffled = {line: i for i, line in enumerate(shuffled_lines)}

    mapping = [line_to_index_shuffled[line] for line in unshuffled_lines]
    return mapping

def reorder_output(processed_file, mapping, output_file):
    with open(processed_file, 'r', encoding='utf-8') as file:
        processed_lines = [line.strip() for line in file.readlines()]

    reordered_lines = [processed_lines[i] for i in mapping]

    with open(output_file, 'w', encoding='utf-8') as file:
        for line in reordered_lines:
            file.write(line + '\n')

if __name__ == "__main__":
    unshuffled_file = 'en-zh.en-filtered-wsd.en'
    shuffled_file = 'en-zh.en-filtered-wsd.en.shuffled'
    processed_file = 'en-zh.en-filtered-wsd-processed.en'
    output_file = 'en-zh.en-filtered-wsd-processed-reordered.en'

    mapping = create_mapping(unshuffled_file, shuffled_file)
    reorder_output(processed_file, mapping, output_file)


## Perform BERT-WSD on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv

3. **Enter the host `slurm`:** 
    ```bash
    srun --pty bash

4. **Find number of CPUs/cores on the machine:**
    ```bash
    nproc --all
(Adjust batch size in `preprocess_file` function to match the number of CPUs)

5. **Run the file to perform WSD using BERT:**
    ```bash
    nice -n 400 python wsd.py

In [ ]:
# Copy this whole code block into wsd.py to run in SoC Computer Cluster
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
import pandas as pd
import re
import os

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

class BertWSDProcessor:
    def __init__(self, model_path='bert-base-uncased', device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()
    
    def identify_ambiguous_words(self, sentence):
        """Identify potentially ambiguous words in the sentence"""
        tokens = word_tokenize(sentence)
        ambiguous_words = []
        
        for token in tokens:
            # Check if the word has multiple senses in WordNet
            synsets = wordnet.synsets(token)
            if len(synsets) > 1:
                ambiguous_words.append(token)
                
        return ambiguous_words
    
    def disambiguate_word(self, word, context):
        """Use BERT-WSD to disambiguate a word in context"""
        # Format input for BERT
        inputs = self.tokenizer(context, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Get predicted sense ID (implementation depends on your specific BERT-WSD model)
        predicted_sense = outputs.logits.argmax().item()
        
        # Map sense ID to WordNet sense (this mapping depends on your model)
        # For simplicity, we'll just return the sense ID in this example
        return f"{word}#{predicted_sense}"
    
    def process_sentence(self, sentence):
        """Process a sentence, disambiguating ambiguous words"""
        ambiguous_words = self.identify_ambiguous_words(sentence)
        processed_sentence = sentence
        
        for word in ambiguous_words:
            # Get the disambiguated sense
            disambiguated_word = self.disambiguate_word(word, sentence)
            
            # Replace the word with its disambiguated form
            # This simple replacement strategy might need improvement for real use cases
            processed_sentence = re.sub(r'\b' + word + r'\b', disambiguated_word, processed_sentence, 1)
            
        return processed_sentence

def preprocess_file(input_file, output_file, batch_size=32, save_interval=1000):
    """Preprocess an entire file using BERT-WSD with resume support."""
    processor = BertWSDProcessor()

    # Check how many lines are already processed
    processed_lines_count = 0
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_lines_count = sum(1 for _ in f)  # Count existing lines
    
    print(f"Resuming from line {processed_lines_count}...")

    # Read input file and skip already processed lines
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[processed_lines_count:]  # Skip processed lines

    processed_lines = []
    
    # Process remaining lines
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        
        for line in batch:
            processed_line = processor.process_sentence(line.strip())
            processed_lines.append(processed_line)
        
        print(f"Processed {processed_lines_count + min(i+batch_size, len(lines))}/{processed_lines_count + len(lines)} lines")

        # Save every `save_interval` lines
        if len(processed_lines) >= save_interval:
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write('\n'.join(processed_lines) + '\n')
            print(f"Saved {len(processed_lines)} lines to {output_file}")
            processed_lines = []  # Clear the buffer

    # Save any remaining lines
    if processed_lines:
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(processed_lines) + '\n')
        print(f"Final save: {len(processed_lines)} lines to {output_file}")

    print(f"Preprocessing complete. Output saved to {output_file}")

if __name__ == "__main__":
    input_file = "./en-zh.en-filtered-wsd.en"
    output_file = "./en-zh.en-filtered-wsd-processed.en"
    
    preprocess_file(input_file, output_file)

In [4]:
# train a sentencepiece model for subwording
!python MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered-wsd-processed.en ./en-zh.zh-filtered-wsd.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered-wsd-processed.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered-wsd-processed.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_ext

In [4]:
# subword the dataset
!python MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered-wsd-processed.en ./en-zh.zh-filtered-wsd.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh.en-filtered-wsd-processed.en
Target Dataset: ./en-zh.zh-filtered-wsd.zh
Done subwording the source file! Output: ./en-zh.en-filtered-wsd-processed.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered-wsd.zh.subword


In [ ]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en && echo "-----" && head -n 3 ./en-zh.zh-filtered-wsd.zh



It's a debate that's still continuing, and it will continue to rumble, because this object is one of the great declarations of a human aspiration.
Especially, it destroys our ability to trust each other, to feel that we're all in the same boat, because it's obvious we're not.
Work doesn't make you happy, does it? Mostly it's tough.
-----
这场辩论仍在继续 它将继续产生轰动的效应 因为赛鲁士圆柱 是象征人类渴望的 伟大的声明之一
更重要的是，它毁掉我们彼此的信任， 毁掉“大家都身处同一条船”的感觉，因为很明显的，我们不在一条船上（因为不平等）。
工作没法使你快乐，不是吗？工作总是辛苦的。


In [5]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered-wsd.zh.subword after

▁en
▁Thank ▁you ▁so # 1 ▁much # 1 , ▁Chris . ▁And ▁it ' s ▁truly # 1 ▁a # 1 ▁great # 1 ▁honor # 1 ▁to ▁have # 1 ▁the ▁opportunity ▁to ▁come # 1 ▁to ▁this ▁stage # 1 ▁twice # 1 ; ▁I # 1 ' m ▁extremely # 1 ▁grateful # 1 .
▁I # 1 # 1 # 1 ▁have # 1 ▁been # 1 ▁blown # 1 ▁away # 1 ▁by # 1 ▁this ▁conference # 1 , ▁and ▁I ▁want # 1 ▁to ▁thank ▁all # 1 ▁of ▁you ▁for ▁the ▁many ▁nice # 1 ▁comment s # 1 ▁about # 1 ▁what ▁I ▁had # 1 ▁to ▁say # 1 ▁the ▁other # 1 ▁night # 1 .
---
==> ./en-zh.zh-filtered-wsd.zh.subword <==
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论
head: after: No such file or directory


In [3]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 100 and 500 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered-wsd-processed.en.subword ./en-zh.zh-filtered-wsd.zh.subword

^C


In [4]:
!wc -l ./*.subword.*

    2000 ./en-zh.en-filtered-wsd-processed.en.subword.dev
    2000 ./en-zh.en-filtered-wsd-processed.en.subword.test
  220753 ./en-zh.en-filtered-wsd-processed.en.subword.train
    2000 ./en-zh.zh-filtered-wsd.zh.subword.dev
    2000 ./en-zh.zh-filtered-wsd.zh.subword.test
  220753 ./en-zh.zh-filtered-wsd.zh.subword.train
  449506 total


In [5]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-wsd-processed.en.subword.train <==
▁It ' s ▁a # 1 # 1 ▁debate # 1 ▁that ' s ▁still # 1 ▁continuing # 1 , ▁and ▁it ▁will # 1 ▁continue # 1 ▁to ▁ rum ble # 1 , ▁because ▁this ▁object # 1 ▁is # 1 ▁one # 1 ▁of ▁the ▁great # 1 ▁de c lar ation s # 1 ▁of ▁a ▁human # 1 ▁aspiration # 1 .

==> ./en-zh.zh-filtered-wsd.zh.subword.train <==
▁ 这场 辩论 仍 在 继续 ▁它 将 继续 产生 轰 动 的 效应 ▁因为 赛 鲁 士 圆 柱 ▁是 象征 人类 渴望 的 ▁ 伟大的 声明 之一

==> ./en-zh.en-filtered-wsd-processed.en.subword.dev <==
▁I # 1 ▁met # 1 ▁Con st ance # 1 ▁O ko ll et , ▁who ▁had # 1 # 1 # 1 ▁formed # 1 ▁a # 1 # 1 ▁women # 1 ' s ▁group # 1 ▁in # 1 # 1 ▁Eastern # 1 ▁Uganda , ▁and ▁she ▁told # 1 ▁me ▁that ▁when ▁she ▁was # 1 ▁growing # 1 ▁up # 1 , ▁she ▁had ▁a ▁very # 1 ▁normal # 1 ▁life # 1 ▁in ▁her ▁village # 1 ▁and ▁they ▁didn ' t ▁go # 1 ▁hungry # 1 , ▁they ▁knew # 1 # 1 # 1 ▁that ▁the ▁season s # 1 ▁would ▁come # 1 # 1 ▁as # 1 ▁they ▁were # 1 ▁predicted # 1 ▁to ▁come , ▁they ▁knew ▁when ▁to ▁so w # 1 ▁and ▁t